# Bloomberg API Examples
This notebook demonstrates connection management, session handling, and data retrieval for Reference, Historical, Intraday, Streaming data, and BQL queries using the Bloomberg Terminal or Desktop API.

In [ ]:
from datetime import date, datetime, timedelta
from Library.Bloomberg import BloombergAPI

# Reference Data
Fetching point-in-time data or static descriptive information for securities.

### Context Manager (Auto-connect/disconnect)
The `with` statement automatically connects to the terminal, opens the necessary services, and gracefully disconnects when finished. It seamlessly handles multiple tickers and fields.

In [ ]:
with BloombergAPI() as bbg:
    df = bbg.reference.fetch(["XBTUSD Curncy", "ETHUSD Curncy"], ["PX_LAST", "NAME"])
    display(df)

### Manual Connection Management
You can also manually `connect()` and `disconnect()` if you need to hold the connection open across multiple independent operations.

In [ ]:
bbg = BloombergAPI().connect()
try:
    df = bbg.reference.fetch("XBTUSD Curncy", "PX_LAST")
    display(df)
finally:
    bbg.disconnect()

# Historical Data
Fetching end-of-day time series data across a specific date range.

### Date Ranges & Timeframes
You can fetch historical data over a specified period. The results are returned as a Polars DataFrame by default, sorted securely by date.

In [ ]:
with BloombergAPI() as bbg:
    df = bbg.historical.fetch("XBTUSD Curncy", "PX_LAST", start=date(2024, 1, 1), stop=date(2024, 3, 31))
    display(df)

### Single Month Example
Demonstrating a shorter historical window utilizing manual connection handling.

In [ ]:
bbg = BloombergAPI().connect()
try:
    df = bbg.historical.fetch("ETHUSD Curncy", "PX_LAST", start=date(2024, 1, 1), stop=date(2024, 1, 31))
    display(df)
finally:
    bbg.disconnect()

# Intraday Data
Fetching granular intraday data, either aggregated into regular bars or as individual discrete ticks.

### Intraday Bars
Fetch aggregated bar data (e.g., 60-minute intervals). You can specify the `event_type` to dictate what the bars represent (e.g., TRADE, BID, or ASK).

In [ ]:
start_dt = datetime.now() - timedelta(days=1)
with BloombergAPI() as bbg:
    # Fetch 60-minute BID bars for the last 24 hours
    df = bbg.intraday.bars("XBTUSD Curncy", start=start_dt, interval=60, event_type="BID")
    display(df)

### Intraday Ticks
Fetch individual tick data for a highly granular view of market activity without any interval aggregation.

In [ ]:
start_dt = datetime.now() - timedelta(days=1)
bbg = BloombergAPI().connect()
try:
    # Fetch individual tick data for the last 24 hours
    df = bbg.intraday.ticks("XBTUSD Curncy", start=start_dt, event_types="BID")
    display(df)
finally:
    bbg.disconnect()

# Streaming Data
Subscribing to real-time, live market data updates directly from the Bloomberg terminal.

### Streaming with Callbacks
Use `subscribe()` to listen for live updates. Pass a custom `callback` function that processes the incoming DataFrames (or dicts). You can optionally set a `limit` to auto-unsubscribe after a certain number of updates are received.

In [ ]:
def on_data(df): 
    display(df)

with BloombergAPI() as bbg:
    # Subscribe to the last price, terminating automatically after receiving 2 updates
    bbg.streaming.subscribe("XBTUSD Curncy", "LAST_PRICE", callback=on_data, limit=2)

### Manual Streaming Lifecycle
If not using the `with` block, ensure you cleanly `disconnect()` to drop all active subscriptions gracefully.

In [ ]:
def on_data(df): 
    display(df)

bbg = BloombergAPI().connect()
try:
    bbg.streaming.subscribe("ETHUSD Curncy", "BID", callback=on_data, limit=2)
finally:
    bbg.disconnect()

# BQL Queries (Commented out)
Executing advanced Bloomberg Query Language (BQL) strings directly.

> **Note:** This module requires a BQL-enabled Bloomberg account and subscription to function successfully.

In [ ]:
# with BloombergAPI() as bbg:
#     bbg.query.execute("get(px_last) for(['XBTUSD Curncy'])")

In [ ]:
# bbg = BloombergAPI().connect()
# try:
#     bbg.query.execute("get(px_last) for(['XBTUSD Curncy'])")
# finally:
#     bbg.disconnect()